# RNN, LSTM, GRU from Scratch

**Goal:** Implement vanilla RNN, LSTM, and GRU cells with forward and backward passes, then train a character-level language model.

## Key Formulas

### Vanilla RNN
$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$
$$y_t = W_{hy} h_t + b_y$$

### LSTM
$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f) \quad \text{(forget gate)}$$
$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i) \quad \text{(input gate)}$$
$$\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c) \quad \text{(candidate)}$$
$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \quad \text{(cell state)}$$
$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o) \quad \text{(output gate)}$$
$$h_t = o_t \odot \tanh(c_t) \quad \text{(hidden state)}$$

### GRU
$$r_t = \sigma(W_r [h_{t-1}, x_t] + b_r) \quad \text{(reset gate)}$$
$$z_t = \sigma(W_z [h_{t-1}, x_t] + b_z) \quad \text{(update gate)}$$
$$\tilde{h}_t = \tanh(W_h [r_t \odot h_{t-1}, x_t] + b_h) \quad \text{(candidate)}$$
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def dsigmoid(s):
    """Derivative of sigmoid given sigmoid output s."""
    return s * (1 - s)

def dtanh(t):
    """Derivative of tanh given tanh output t."""
    return 1 - t ** 2

def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

## 1. Vanilla RNN Cell

In [ ]:
class VanillaRNNCell:
    """Single RNN cell: h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)"""
    
    def __init__(self, input_size, hidden_size):
        scale = np.sqrt(2.0 / (input_size + hidden_size))
        self.W_xh = np.random.randn(input_size, hidden_size) * scale
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale
        self.b_h = np.zeros(hidden_size)
        self.hidden_size = hidden_size
    
    def forward(self, x_t, h_prev):
        """Single step forward.
        Args: x_t (batch, input_size), h_prev (batch, hidden_size)
        Returns: h_t (batch, hidden_size)
        """
        self.x_t = x_t
        self.h_prev = h_prev
        self.z = x_t @ self.W_xh + h_prev @ self.W_hh + self.b_h
        self.h_t = np.tanh(self.z)
        return self.h_t
    
    def backward(self, dh_t):
        """Single step backward.
        Args: dh_t (batch, hidden_size) — gradient from above + from future step
        Returns: dx_t, dh_prev, dW_xh, dW_hh, db_h
        """
        # Through tanh
        dz = dh_t * dtanh(self.h_t)  # (batch, hidden_size)
        
        # Parameter gradients
        dW_xh = self.x_t.T @ dz
        dW_hh = self.h_prev.T @ dz
        db_h = dz.sum(axis=0)
        
        # Input gradients
        dx_t = dz @ self.W_xh.T
        dh_prev = dz @ self.W_hh.T
        
        return dx_t, dh_prev, dW_xh, dW_hh, db_h

# Quick test
rnn_cell = VanillaRNNCell(input_size=10, hidden_size=20)
x_t = np.random.randn(1, 10)
h_prev = np.zeros((1, 20))
h_t = rnn_cell.forward(x_t, h_prev)
print(f"h_t shape: {h_t.shape}, range: [{h_t.min():.3f}, {h_t.max():.3f}]")

## 2. LSTM Cell

The LSTM solves the vanishing gradient problem by introducing a **cell state** $c_t$ that can carry information across many time steps via additive updates (rather than multiplicative).

- **Forget gate** $f_t$: what to erase from cell state
- **Input gate** $i_t$: what new information to write
- **Output gate** $o_t$: what to expose to the next layer

In [ ]:
class LSTMCell:
    """LSTM cell with forget, input, output gates."""
    
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        concat_size = input_size + hidden_size
        scale = np.sqrt(2.0 / concat_size)
        
        # Combined weight matrix for all 4 gates: [f, i, c_tilde, o]
        # Shape: (input_size + hidden_size, 4 * hidden_size)
        self.W = np.random.randn(concat_size, 4 * hidden_size) * scale
        self.b = np.zeros(4 * hidden_size)
        # Initialize forget gate bias to 1 (important trick)
        self.b[:hidden_size] = 1.0
    
    def forward(self, x_t, h_prev, c_prev):
        """Single LSTM step.
        Args:
            x_t: (batch, input_size)
            h_prev: (batch, hidden_size)
            c_prev: (batch, hidden_size)
        Returns:
            h_t, c_t
        """
        H = self.hidden_size
        self.concat = np.concatenate([h_prev, x_t], axis=1)  # (batch, concat_size)
        self.c_prev = c_prev
        
        # All gates in one matmul
        gates = self.concat @ self.W + self.b  # (batch, 4H)
        
        # Split and activate
        self.f = sigmoid(gates[:, :H])          # forget gate
        self.i = sigmoid(gates[:, H:2*H])       # input gate
        self.c_tilde = np.tanh(gates[:, 2*H:3*H])  # candidate
        self.o = sigmoid(gates[:, 3*H:])        # output gate
        
        # Cell state and hidden state
        self.c_t = self.f * c_prev + self.i * self.c_tilde
        self.tanh_c = np.tanh(self.c_t)
        self.h_t = self.o * self.tanh_c
        
        return self.h_t, self.c_t
    
    def backward(self, dh_t, dc_t):
        """Single LSTM step backward.
        Args:
            dh_t: (batch, H) gradient w.r.t. h_t
            dc_t: (batch, H) gradient w.r.t. c_t (from future step)
        Returns:
            dx_t, dh_prev, dc_prev, dW, db
        """
        H = self.hidden_size
        
        # Gradient through h_t = o * tanh(c_t)
        do = dh_t * self.tanh_c
        dc_total = dh_t * self.o * dtanh(self.tanh_c) + dc_t
        
        # Gradient through cell state update
        df = dc_total * self.c_prev
        di = dc_total * self.c_tilde
        dc_tilde = dc_total * self.i
        dc_prev = dc_total * self.f
        
        # Through activations
        df_raw = df * dsigmoid(self.f)
        di_raw = di * dsigmoid(self.i)
        dc_tilde_raw = dc_tilde * dtanh(self.c_tilde)
        do_raw = do * dsigmoid(self.o)
        
        # Stack gate gradients
        dgates = np.concatenate([df_raw, di_raw, dc_tilde_raw, do_raw], axis=1)  # (batch, 4H)
        
        # Parameter gradients
        dW = self.concat.T @ dgates
        db = dgates.sum(axis=0)
        
        # Input gradients
        dconcat = dgates @ self.W.T
        dh_prev = dconcat[:, :H]
        dx_t = dconcat[:, H:]
        
        return dx_t, dh_prev, dc_prev, dW, db

# Test
lstm_cell = LSTMCell(input_size=10, hidden_size=20)
h_prev = np.zeros((1, 20))
c_prev = np.zeros((1, 20))
h_t, c_t = lstm_cell.forward(x_t, h_prev, c_prev)
print(f"h_t shape: {h_t.shape}, c_t shape: {c_t.shape}")
print(f"h_t range: [{h_t.min():.3f}, {h_t.max():.3f}]")

## 3. GRU Cell

The GRU is a simplified version of LSTM with only 2 gates (reset $r$ and update $z$), and no separate cell state. Empirically, it often performs comparably to LSTM with fewer parameters.

In [ ]:
class GRUCell:
    """GRU cell with reset and update gates."""
    
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        concat_size = input_size + hidden_size
        scale = np.sqrt(2.0 / concat_size)
        
        # Weights for reset and update gates (computed together)
        self.W_rz = np.random.randn(concat_size, 2 * hidden_size) * scale
        self.b_rz = np.zeros(2 * hidden_size)
        
        # Weights for candidate hidden state
        self.W_h = np.random.randn(concat_size, hidden_size) * scale
        self.b_h = np.zeros(hidden_size)
    
    def forward(self, x_t, h_prev):
        """Single GRU step."""
        H = self.hidden_size
        self.x_t = x_t
        self.h_prev = h_prev
        
        # Compute reset and update gates
        self.concat_rz = np.concatenate([h_prev, x_t], axis=1)
        rz = self.concat_rz @ self.W_rz + self.b_rz
        self.r = sigmoid(rz[:, :H])     # reset gate
        self.z = sigmoid(rz[:, H:])     # update gate
        
        # Candidate hidden state
        self.concat_h = np.concatenate([self.r * h_prev, x_t], axis=1)
        self.h_tilde = np.tanh(self.concat_h @ self.W_h + self.b_h)
        
        # Final hidden state
        self.h_t = (1 - self.z) * h_prev + self.z * self.h_tilde
        return self.h_t
    
    def backward(self, dh_t):
        """Single GRU step backward."""
        H = self.hidden_size
        
        # Through: h_t = (1-z)*h_prev + z*h_tilde
        dh_prev_direct = dh_t * (1 - self.z)
        dz = dh_t * (self.h_tilde - self.h_prev)
        dh_tilde = dh_t * self.z
        
        # Through tanh for h_tilde
        dh_tilde_raw = dh_tilde * dtanh(self.h_tilde)
        dW_h = self.concat_h.T @ dh_tilde_raw
        db_h = dh_tilde_raw.sum(axis=0)
        dconcat_h = dh_tilde_raw @ self.W_h.T
        
        # Split dconcat_h -> d(r*h_prev), dx_t_part1
        dr_h_prev = dconcat_h[:, :H]
        dx_t_1 = dconcat_h[:, H:]
        
        dr = dr_h_prev * self.h_prev
        dh_prev_from_r = dr_h_prev * self.r
        
        # Through sigmoid for r and z
        dr_raw = dr * dsigmoid(self.r)
        dz_raw = dz * dsigmoid(self.z)
        drz_raw = np.concatenate([dr_raw, dz_raw], axis=1)
        
        dW_rz = self.concat_rz.T @ drz_raw
        db_rz = drz_raw.sum(axis=0)
        dconcat_rz = drz_raw @ self.W_rz.T
        
        dh_prev_from_rz = dconcat_rz[:, :H]
        dx_t_2 = dconcat_rz[:, H:]
        
        dh_prev = dh_prev_direct + dh_prev_from_r + dh_prev_from_rz
        dx_t = dx_t_1 + dx_t_2
        
        return dx_t, dh_prev, dW_rz, db_rz, dW_h, db_h

# Test
gru_cell = GRUCell(input_size=10, hidden_size=20)
h_t_gru = gru_cell.forward(x_t, np.zeros((1, 20)))
print(f"GRU h_t shape: {h_t_gru.shape}, range: [{h_t_gru.min():.3f}, {h_t_gru.max():.3f}]")

## 4. BPTT — Backpropagation Through Time

Unroll the RNN for $T$ steps and backpropagate through the entire sequence. Gradients accumulate across all time steps.

In [ ]:
class VanillaRNN:
    """Full Vanilla RNN with BPTT for sequence-to-sequence (e.g., char LM)."""
    
    def __init__(self, vocab_size, hidden_size):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        scale_xh = np.sqrt(2.0 / (vocab_size + hidden_size))
        scale_hh = np.sqrt(2.0 / (hidden_size + hidden_size))
        scale_hy = np.sqrt(2.0 / (hidden_size + vocab_size))
        
        self.W_xh = np.random.randn(vocab_size, hidden_size) * scale_xh
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale_hh
        self.b_h = np.zeros(hidden_size)
        self.W_hy = np.random.randn(hidden_size, vocab_size) * scale_hy
        self.b_y = np.zeros(vocab_size)
        
        # Adagrad memory
        self.mW_xh = np.zeros_like(self.W_xh)
        self.mW_hh = np.zeros_like(self.W_hh)
        self.mb_h = np.zeros_like(self.b_h)
        self.mW_hy = np.zeros_like(self.W_hy)
        self.mb_y = np.zeros_like(self.b_y)
    
    def forward(self, inputs, h_prev):
        """Forward pass through time.
        Args:
            inputs: list of int (character indices), length T
            h_prev: (1, hidden_size)
        Returns:
            probs: dict t -> (1, vocab_size) softmax probs
            h_last: final hidden state
        """
        self.inputs = inputs
        self.xs = {}   # one-hot inputs
        self.hs = {-1: h_prev.copy()}
        self.probs = {}
        
        for t, idx in enumerate(inputs):
            self.xs[t] = np.zeros((1, self.vocab_size))
            self.xs[t][0, idx] = 1.0
            self.hs[t] = np.tanh(
                self.xs[t] @ self.W_xh + self.hs[t-1] @ self.W_hh + self.b_h
            )
            logits = self.hs[t] @ self.W_hy + self.b_y
            self.probs[t] = softmax(logits)
        
        return self.probs, self.hs[len(inputs) - 1]
    
    def loss(self, targets):
        """Cross-entropy loss."""
        total = 0
        for t, tgt in enumerate(targets):
            total += -np.log(self.probs[t][0, tgt] + 1e-12)
        return total / len(targets)
    
    def backward(self, targets):
        """BPTT. Returns gradients."""
        T = len(targets)
        dW_xh = np.zeros_like(self.W_xh)
        dW_hh = np.zeros_like(self.W_hh)
        db_h = np.zeros_like(self.b_h)
        dW_hy = np.zeros_like(self.W_hy)
        db_y = np.zeros_like(self.b_y)
        dh_next = np.zeros((1, self.hidden_size))
        
        for t in reversed(range(T)):
            # Softmax gradient
            dy = self.probs[t].copy()
            dy[0, targets[t]] -= 1.0
            dy /= T
            
            dW_hy += self.hs[t].T @ dy
            db_y += dy[0]
            
            dh = dy @ self.W_hy.T + dh_next
            dh_raw = dh * dtanh(self.hs[t])
            
            dW_xh += self.xs[t].T @ dh_raw
            dW_hh += self.hs[t-1].T @ dh_raw
            db_h += dh_raw[0]
            
            dh_next = dh_raw @ self.W_hh.T
        
        # Gradient clipping
        for grad in [dW_xh, dW_hh, db_h, dW_hy, db_y]:
            np.clip(grad, -5, 5, out=grad)
        
        return dW_xh, dW_hh, db_h, dW_hy, db_y
    
    def step(self, targets, lr=0.01):
        """One training step with Adagrad."""
        grads = self.backward(targets)
        params = [self.W_xh, self.W_hh, self.b_h, self.W_hy, self.b_y]
        mems = [self.mW_xh, self.mW_hh, self.mb_h, self.mW_hy, self.mb_y]
        
        for param, grad, mem in zip(params, grads, mems):
            mem += grad * grad
            param -= lr * grad / (np.sqrt(mem) + 1e-8)

print("VanillaRNN class defined.")

## 5. Character-Level Language Model

In [ ]:
# Training data
text = "hello world hello neural network hello deep learning hello recurrent network "
text = text * 10  # repeat for more data

chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}

data = [char_to_idx[c] for c in text]
print(f"Vocab size: {vocab_size}, chars: {''.join(chars)}")
print(f"Data length: {len(data)}")

In [ ]:
def sample(rnn, h, seed_idx, n=100):
    """Sample n characters from the model."""
    x_idx = seed_idx
    result = []
    for _ in range(n):
        x_onehot = np.zeros((1, rnn.vocab_size))
        x_onehot[0, x_idx] = 1.0
        h = np.tanh(x_onehot @ rnn.W_xh + h @ rnn.W_hh + rnn.b_h)
        logits = h @ rnn.W_hy + rnn.b_y
        p = softmax(logits)[0]
        x_idx = np.random.choice(rnn.vocab_size, p=p)
        result.append(idx_to_char[x_idx])
    return ''.join(result)

# Train
rnn = VanillaRNN(vocab_size, hidden_size=64)
seq_len = 25
losses = []
smooth_loss = -np.log(1.0 / vocab_size)  # initial loss

n_iters = 2000
pointer = 0
h_state = np.zeros((1, 64))

for iteration in range(n_iters):
    if pointer + seq_len + 1 >= len(data):
        pointer = 0
        h_state = np.zeros((1, 64))
    
    inputs = data[pointer:pointer + seq_len]
    targets = data[pointer + 1:pointer + seq_len + 1]
    
    probs, h_state = rnn.forward(inputs, h_state)
    loss = rnn.loss(targets)
    rnn.step(targets, lr=0.01)
    
    smooth_loss = 0.999 * smooth_loss + 0.001 * loss
    losses.append(smooth_loss)
    
    if (iteration + 1) % 500 == 0:
        print(f"Iter {iteration+1}: loss={smooth_loss:.3f}")
        print(f"  Sample: {sample(rnn, h_state, inputs[0], 60)}")
    
    pointer += seq_len

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses)
ax.set_xlabel('Iteration')
ax.set_ylabel('Smoothed Loss')
ax.set_title('Vanilla RNN Character LM Training')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Visualize Gate Activations (LSTM)

Run a short sequence through the LSTM and plot how each gate activates.

In [ ]:
# Create an LSTM and run a sequence
lstm = LSTMCell(input_size=vocab_size, hidden_size=32)
test_seq = [char_to_idx[c] for c in "hello world "]

h = np.zeros((1, 32))
c = np.zeros((1, 32))
forget_gates = []
input_gates = []
output_gates = []

for idx in test_seq:
    x_onehot = np.zeros((1, vocab_size))
    x_onehot[0, idx] = 1.0
    h, c = lstm.forward(x_onehot, h, c)
    forget_gates.append(lstm.f[0].copy())
    input_gates.append(lstm.i[0].copy())
    output_gates.append(lstm.o[0].copy())

forget_gates = np.array(forget_gates)  # (T, H)
input_gates = np.array(input_gates)
output_gates = np.array(output_gates)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
char_labels = list("hello world ")

# Show first 8 hidden units
n_show = 8
for ax, gate_data, title in zip(axes, 
    [forget_gates, input_gates, output_gates],
    ['Forget Gate', 'Input Gate', 'Output Gate']):
    im = ax.imshow(gate_data[:, :n_show].T, aspect='auto', cmap='RdYlBu_r', vmin=0, vmax=1)
    ax.set_yticks(range(n_show))
    ax.set_yticklabels([f'h{i}' for i in range(n_show)])
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.02)

axes[-1].set_xticks(range(len(char_labels)))
axes[-1].set_xticklabels(char_labels, fontsize=12)
axes[-1].set_xlabel('Input Character')
plt.suptitle('LSTM Gate Activations (first 8 hidden units)', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Gradient Norms Over Time Steps

Demonstrate why vanilla RNNs suffer from vanishing/exploding gradients.

In [ ]:
def compute_gradient_norms_rnn(W_hh, T=50):
    """Simulate gradient flow: dh_1/dh_T through a chain of W_hh and tanh."""
    H = W_hh.shape[0]
    # Forward pass to get hidden states
    h = np.random.randn(H) * 0.1
    hs = [h]
    for t in range(T):
        h = np.tanh(W_hh @ h)
        hs.append(h)
    
    # Backward: compute ||dh_t/dh_T|| for each t
    norms = []
    grad = np.eye(H)
    for t in range(T-1, -1, -1):
        diag = np.diag(dtanh(hs[t+1]))  # tanh derivative
        grad = diag @ W_hh @ grad        # Jacobian chain rule, but we track norm
        norms.append(np.linalg.norm(grad))
    
    return norms[::-1]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Small spectral norm -> vanishing
W_small = np.random.randn(20, 20) * 0.3
norms_small = compute_gradient_norms_rnn(W_small, T=50)
axes[0].semilogy(norms_small)
axes[0].set_title('Small weights -> Vanishing')
axes[0].set_xlabel('Time step (from end)')
axes[0].set_ylabel('Gradient norm (log scale)')

# Large spectral norm -> exploding
W_large = np.random.randn(20, 20) * 1.5
norms_large = compute_gradient_norms_rnn(W_large, T=50)
axes[1].semilogy(norms_large)
axes[1].set_title('Large weights -> Exploding')
axes[1].set_xlabel('Time step (from end)')

# Orthogonal initialization -> stable
W_ortho = np.linalg.qr(np.random.randn(20, 20))[0]
norms_ortho = compute_gradient_norms_rnn(W_ortho, T=50)
axes[2].semilogy(norms_ortho)
axes[2].set_title('Orthogonal init -> Stable')
axes[2].set_xlabel('Time step (from end)')

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Numerical Gradient Check for LSTM

In [ ]:
# Gradient check for the LSTM cell
lstm_check = LSTMCell(input_size=5, hidden_size=4)
x_check = np.random.randn(1, 5)
h_check = np.random.randn(1, 4)
c_check = np.random.randn(1, 4)

# Forward
h_out, c_out = lstm_check.forward(x_check, h_check, c_check)
# Use sum of outputs as scalar loss
dh = np.ones_like(h_out)
dc = np.zeros_like(c_out)

dx_anal, dh_prev_anal, dc_prev_anal, dW_anal, db_anal = lstm_check.backward(dh, dc)

# Numerical gradient for W
eps = 1e-5
dW_num = np.zeros_like(lstm_check.W)
for i in range(min(3, lstm_check.W.shape[0])):
    for j in range(min(3, lstm_check.W.shape[1])):
        W_save = lstm_check.W[i, j]
        lstm_check.W[i, j] = W_save + eps
        h_plus, _ = lstm_check.forward(x_check, h_check, c_check)
        lstm_check.W[i, j] = W_save - eps
        h_minus, _ = lstm_check.forward(x_check, h_check, c_check)
        lstm_check.W[i, j] = W_save
        dW_num[i, j] = np.sum(dh * (h_plus - h_minus)) / (2 * eps)

# Restore forward state
lstm_check.forward(x_check, h_check, c_check)

print("LSTM Gradient Check (first 3x3 block of dW):")
print(f"Analytical:\n{dW_anal[:3, :3]}")
print(f"Numerical:\n{dW_num[:3, :3]}")
rel_err = np.abs(dW_anal[:3, :3] - dW_num[:3, :3]) / (np.abs(dW_num[:3, :3]) + 1e-8)
print(f"Max relative error: {rel_err.max():.2e}")

## 9. Numerical Gradient Check for GRU

In [ ]:
# Gradient check for the GRU cell
gru_check = GRUCell(input_size=5, hidden_size=4)
x_check = np.random.randn(1, 5)
h_check = np.random.randn(1, 4)

# Forward
h_out = gru_check.forward(x_check, h_check)
dh = np.ones_like(h_out)

dx_anal, dh_prev_anal, dW_rz_anal, db_rz_anal, dW_h_anal, db_h_anal = gru_check.backward(dh)

# Numerical gradient for W_rz
eps = 1e-5
dW_rz_num = np.zeros_like(gru_check.W_rz)
for i in range(min(3, gru_check.W_rz.shape[0])):
    for j in range(min(3, gru_check.W_rz.shape[1])):
        W_save = gru_check.W_rz[i, j]
        gru_check.W_rz[i, j] = W_save + eps
        h_plus = gru_check.forward(x_check, h_check)
        gru_check.W_rz[i, j] = W_save - eps
        h_minus = gru_check.forward(x_check, h_check)
        gru_check.W_rz[i, j] = W_save
        dW_rz_num[i, j] = np.sum(dh * (h_plus - h_minus)) / (2 * eps)

# Numerical gradient for W_h
dW_h_num = np.zeros_like(gru_check.W_h)
for i in range(min(3, gru_check.W_h.shape[0])):
    for j in range(min(3, gru_check.W_h.shape[1])):
        W_save = gru_check.W_h[i, j]
        gru_check.W_h[i, j] = W_save + eps
        h_plus = gru_check.forward(x_check, h_check)
        gru_check.W_h[i, j] = W_save - eps
        h_minus = gru_check.forward(x_check, h_check)
        gru_check.W_h[i, j] = W_save
        dW_h_num[i, j] = np.sum(dh * (h_plus - h_minus)) / (2 * eps)

# Restore forward state
gru_check.forward(x_check, h_check)

print("GRU Gradient Check:")
print(f"
dW_rz (first 3x3 block):")
print(f"Analytical:
{dW_rz_anal[:3, :3]}")
print(f"Numerical:
{dW_rz_num[:3, :3]}")
rel_err_rz = np.abs(dW_rz_anal[:3, :3] - dW_rz_num[:3, :3]) / (np.abs(dW_rz_num[:3, :3]) + 1e-8)
print(f"Max relative error: {rel_err_rz.max():.2e}")

print(f"
dW_h (first 3x3 block):")
print(f"Analytical:
{dW_h_anal[:3, :3]}")
print(f"Numerical:
{dW_h_num[:3, :3]}")
rel_err_h = np.abs(dW_h_anal[:3, :3] - dW_h_num[:3, :3]) / (np.abs(dW_h_num[:3, :3]) + 1e-8)
print(f"Max relative error: {rel_err_h.max():.2e}")

## Interview Questions

### Q: "Why LSTM over vanilla RNN?"

**Vanishing gradients.** In a vanilla RNN, gradients are multiplied by $W_{hh}$ at each time step during BPTT. If the spectral radius of $W_{hh}$ is < 1, gradients vanish exponentially; if > 1, they explode.

LSTM fixes this with the **cell state** $c_t$, which has an **additive** update path ($c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$). The forget gate allows gradients to flow through unchanged ($f_t \approx 1$), creating a "gradient highway."

### Q: "What does each LSTM gate do?"

| Gate | Formula | Purpose |
|------|---------|--------|
| **Forget** $f_t$ | $\sigma(W_f [h_{t-1}, x_t])$ | Decides what to erase from cell state |
| **Input** $i_t$ | $\sigma(W_i [h_{t-1}, x_t])$ | Decides what new info to write |
| **Output** $o_t$ | $\sigma(W_o [h_{t-1}, x_t])$ | Decides what to expose as hidden state |

The **candidate** $\tilde{c}_t = \tanh(\ldots)$ proposes new content; the input gate scales it.

### Q: "GRU vs LSTM?"

| | LSTM | GRU |
|---|------|-----|
| Gates | 3 (forget, input, output) | 2 (reset, update) |
| Cell state | Separate $c_t$ and $h_t$ | Only $h_t$ |
| Parameters | More ($4 \times$ hidden projections) | Fewer ($3 \times$ hidden projections) |
| Performance | Slightly better on longer sequences | Comparable; faster to train |

In practice, try both. GRU is a good default when compute is limited.

### Q: "What is gradient clipping and why is it needed?"

Gradient clipping caps the norm of the gradient vector to prevent exploding gradients. Two variants:
- **Value clipping**: `clip(g, -threshold, threshold)` (element-wise)
- **Norm clipping**: `g = g * threshold / max(||g||, threshold)` (scales entire vector)

Norm clipping is preferred as it preserves gradient direction.